# A bit about MLIR
(What all we need)

![Image](https://codeplay.com/assets/images/blogs/2024-02-09-experiences-building-an-mlir-based-sycl-compiler/abstraction-gap-mlir.png?utm_source=chatgpt.com)


![Image](https://user-images.githubusercontent.com/10148468/71177417-f78e4d80-2239-11ea-92ef-700f42ea503f.png?utm_source=chatgpt.com)

![Image](https://mlir.llvm.org/includes/img/DefUseChains.svg?utm_source=chatgpt.com)

![Image](https://raw.githubusercontent.com/hanchenye/scalehls/master/docs/ScaleHLS.svg?utm_source=chatgpt.com)

Below is a **compact but complete mental model of MLIR**—what it is, how it’s structured, and how it’s used in real compilers (including quantum / fault-tolerant ones).

---

## What MLIR actually is (one-sentence definition)

**MLIR (Multi-Level Intermediate Representation)** is a compiler infrastructure that lets you represent a program **at multiple abstraction levels at once**, using **custom dialects**, and progressively **lower** them through well-defined passes until you reach executable code.

Think of it as a **compiler construction kit**, not a language.

---

## Why MLIR was created (the core problem it solves)

Traditional compilers look like:

```
Language → One Big IR → Machine Code
```

This breaks down for modern systems:

* ML models
* GPUs / TPUs
* Heterogeneous hardware
* Domain-specific semantics (AI, quantum, DSP, crypto)
* Long lowering chains with constraints

MLIR instead assumes:

> “There is no single perfect IR.”

So it lets you have **many IRs**, connected by **structured lowering paths**.

---

## Core concepts (this is the heart of MLIR)

### 1. Dialects

A **dialect** is a *namespace + set of operations + types + rules* for a domain.

Examples (real-world):

* `arith` → integer/float math
* `scf` → structured control flow (`for`, `if`)
* `affine` → loop nests with static bounds
* `gpu` → GPU kernels
* `llvm` → low-level, LLVM-compatible ops

You can also define your own:

* `qc` → quantum circuits
* `ft` → fault-tolerant logical operations
* `pulse` → hardware control primitives

👉 Dialects let you **encode semantics explicitly**, instead of hiding them in comments or conventions.

---

### 2. Operations (Ops)

Everything in MLIR is an **operation**.

An op has:

* operands (inputs)
* results (outputs)
* attributes (compile-time metadata)
* regions (nested blocks of ops)
* verification rules

Example (conceptual):

```
ft.t %q : !ft.logical_qubit
```

This is *much richer* than a flat instruction like LLVM IR.

---

### 3. Types (first-class, extensible)

MLIR types are not limited to `i32`, `float`, etc.

You can define:

* `!q.qubit`
* `!ft.logical_qubit<code=surface, d=15>`
* `!tensor<128x128xf16>`
* `!memref<1024xf32>`

Types carry **semantic meaning**, and passes can reason over them.

---

### 4. Regions & SSA (structured IR)

MLIR uses SSA like LLVM, but with **nested regions**.

This enables:

* structured control flow
* scoping
* compositional lowering
* embedding one IR inside another

This is why MLIR works well for:

* loops
* kernels
* quantum subcircuits
* schedules

---

## How MLIR is *used* in a compiler (end-to-end)

### Step 1: Frontend → High-level dialect

You parse or import a program into a **high-level dialect**.

Examples:

* Python → `linalg` / `tensor`
* QASM → `qc`
* DSL → custom dialect

Goal:

* Preserve intent
* Enable domain-specific optimizations

---

### Step 2: Analysis + Optimization at the right level

Instead of flattening early, MLIR lets you optimize **where it makes sense**:

* algebraic simplification at high level
* loop fusion at affine level
* scheduling later
* hardware constraints last

This avoids “premature lowering,” which kills optimization opportunities.

---

### Step 3: Progressive lowering (dialect → dialect)

Lowering is done via **passes**:

```
qc → ft → schedule → llvm
```

Each pass:

* replaces ops with simpler/more concrete ones
* preserves correctness via verifiers
* may attach metadata (costs, timing, resources)

Crucially:

> Multiple dialects can coexist in one module during lowering.

---

### Step 4: Final lowering to LLVM IR or target backend

Eventually, MLIR lowers to:

* LLVM IR
* GPU binaries
* custom runtimes
* simulators

MLIR doesn’t replace LLVM—it **feeds into it**.

---

## Passes: the engine of MLIR

MLIR provides a rich pass system:

* **Rewrite passes** (pattern-based)
* **Canonicalization**
* **Common subexpression elimination**
* **Dead code elimination**
* **Analysis passes** (costs, dominance, liveness)

You can:

* enforce invariants
* reject illegal transformations
* annotate IR with analysis results

This is ideal for **fault tolerance**, where invariants matter.

---

## Verification & correctness (often overlooked, very important)

Each op can define:

* structural constraints
* type constraints
* semantic invariants

If a pass violates them → compilation fails early.

This makes MLIR suitable for **high-assurance compilers**, not just performance compilers.

---

## How MLIR differs from LLVM IR (clear comparison)

| Aspect               | LLVM IR        | MLIR        |
| -------------------- | -------------- | ----------- |
| Abstraction level    | Low            | Any level   |
| Domain semantics     | Hard to encode | First-class |
| Types                | Fixed          | Extensible  |
| Control flow         | Unstructured   | Structured  |
| Dialects             | ❌              | ✅           |
| Multi-stage lowering | Awkward        | Native      |
| FT / AI / Quantum    | Painful        | Natural     |

MLIR is *above* LLVM, not a replacement.

---

## Where MLIR is used today (credibility check)

MLIR underpins:

* TensorFlow / XLA
* PyTorch 2.x (torch-mlir)
* IREE
* StableHLO
* GPU compilers
* DSP pipelines
* Experimental quantum compilers

It’s production-grade, not academic.

---

## Why MLIR makes sense for **fault-tolerant quantum compilers**

Because FT compilation:

* has **multiple abstraction layers**
* needs **semantic guarantees**
* requires **resource-aware optimization**
* must avoid illegal transformations

MLIR gives you:

* explicit logical vs physical separation
* verifiable invariants
* honest cost accounting
* clean backend targeting

In short:

> MLIR matches the *structure of the problem*, not just the syntax.

---

## TL;DR mental model

* MLIR = **IR framework**, not a language
* Dialects = **domain-specific IRs**
* Passes = **controlled semantic lowering**
* Types & attributes = **meaning, not just bits**
* LLVM = **final backend**, not the whole compiler

---

If you want, next we can:

* design a **minimal MLIR dialect for fault tolerance**
* map **QASM/Qiskit → MLIR**
* compare MLIR vs custom IR for your use case
* or sketch how CDAC-style fault-tolerant transpilers should structure passes




## Proposed leveraging MLIR for Fault Tolerant Implementation
(A comparison between QASM and MLIR)

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

# Step 1: Build a simple circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Step 2: Connect to IBM Quantum cloud
service = QiskitRuntimeService(channel="ibm_quantum_platform")

# Step 3: Choose backend (real quantum hardware)
backend = service.backend("ibmq_quito")

# Step 4: Transpile circuit for hardware
tqc = transpile(qc, backend)

# Step 5: Run using Sampler primitive
sampler = Sampler(backend=backend)
result = sampler.run([tqc]).result()

print(result)


In [7]:
# Diagnostic: Check which Python environment the kernel is using
import sys
import subprocess

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("\nInstalling qiskit in the kernel's Python environment...")
print("=" * 60)

# Install qiskit using the kernel's Python to ensure it's in the right environment
# This ensures qiskit is installed in the same Python that the kernel is using
subprocess.check_call([sys.executable, "-m", "pip", "install", "qiskit-ibm-runtime"])

Python executable: C:\Users\vmuno\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]

Installing qiskit in the kernel's Python environment...


0

# Using MLIR for Fault-Tolerant Quantum Compilation: A Comparative Study with QASM

## Prologue

TL;DR: Fault-tolerant quantum computing introduces concepts—logical qubits, error-correcting codes, syndrome extraction, and classical feedback—that cannot be cleanly represented using flat, gate-level languages like QASM. As quantum hardware scales, compilation becomes a multi-level, hardware-aware optimization problem rather than a simple circuit translation task. This notebook argues that MLIR provides a more suitable foundation for fault-tolerant quantum compilation by making fault tolerance an explicit, first-class compiler abstraction, enabling structured reasoning, late-stage optimization, and portability across quantum architectures.

Quantum computing has reached a stage where raw increases in qubit count no longer translate into meaningful computational capability. Contemporary quantum hardware is dominated by noise, decoherence, and correlated errors, making **fault tolerance** not an optional enhancement but a fundamental requirement for scalable quantum computation.

While quantum programming languages and intermediate representations such as QASM have played a critical role in describing and executing near-term quantum circuits, they were not designed with fault tolerance as a first-class concern. In practice, fault-tolerant quantum computation introduces concepts that fundamentally exceed the expressive power of flat, gate-level descriptions: logical qubits encoded across many physical qubits, error-correcting codes with tunable parameters, syndrome extraction cycles, classical decoding and feedback, and hardware-dependent constraints on error propagation.

As a result, modern fault-tolerant compilation is no longer a simple translation problem from algorithm to hardware instructions. It is a **multi-level optimization problem** that must preserve semantic intent across progressively lower abstractions while making late, hardware-aware decisions about error correction, scheduling, and resource allocation.

This notebook explores the use of **Multi-Level Intermediate Representation (MLIR)** as a foundation for building a **fault-tolerant quantum compiler**, and contrasts it with QASM-based approaches. Rather than presenting MLIR as a replacement for QASM, this work frames MLIR as a compiler-native infrastructure capable of representing fault tolerance explicitly, structurally, and analyzably—capabilities that are essential for scalable, hardware-portable quantum compilation.

The goal of this document is not to propose a finalized standard, but to demonstrate how fault tolerance can be elevated from an implicit backend concern to a **first-class compiler abstraction**, and why such an elevation becomes unavoidable as quantum systems transition from experimental devices to fault-tolerant machines.


## Table of Contents

1. [Abstract / Executive Summary](#1-abstract--executive-summary)

2. [Background & Motivation](#2-background--motivation) <br>
   2.1. [Why Fault-Tolerant Compilation is Hard](#21-why-fault-tolerant-compilation-is-hard) <br>
   2.2. [What an Intermediate Representation Must Support for Fault Tolerance](#22-what-an-intermediate-representation-must-support-for-fault-tolerance) <br>

3. [QASM as an Intermediate Representation](#3-qasm-as-an-intermediate-representation) <br>
   3.1. [What QASM Is Designed For](#31-what-qasm-is-designed-for) <br>
   3.2. [Example: Logical Circuit in QASM](#32-example-logical-circuit-in-qasm) <br>
   3.3. [Attempting Fault Tolerance in QASM](#33-attempting-fault-tolerance-in-qasm)

4. [MLIR: A Compiler-Native Intermediate Representation](#4-mlir-a-compiler-native-intermediate-representation) <br>
   4.1. [What MLIR Is (Conceptual Overview)](#41-what-mlir-is-conceptual-overview) <br>
   4.2. [Why MLIR Is a Better Fit for Fault Tolerance](#42-why-mlir-is-a-better-fit-for-fault-tolerance) 

5. [Designing a Fault-Tolerant Quantum MLIR Stack](#5-designing-a-fault-tolerant-quantum-mlir-stack) <br>
   5.1. [Dialect Stack Overview](#51-dialect-stack-overview) <br>
   5.2. [Logical Dialect](#52-logical-dialect) <br>
   5.3. [Encoded / Fault-Tolerant Dialect](#53-encoded--fault-tolerant-dialect) <br>
   5.4. [Syndrome & Error-Correction Dialect](#54-syndrome--error-correction-dialect) <br>
   5.5. [Physical Dialect](#55-physical-dialect) <br>

6. [Comparative Analysis: MLIR vs QASM](#6-comparative-analysis-mlir-vs-qasm) <br>
   6.1. [Expressiveness](#61-expressiveness) <br>
   6.2. [Optimization Opportunities](#62-optimization-opportunities) <br>
   6.3. [Compiler Passes Enabled](#63-compiler-passes-enabled)

7. [Case Study: Fault-Tolerant Compilation Workflow](#7-case-study-fault-tolerant-compilation-workflow) <br>
   7.1. [Logical Circuit Description](#71-logical-circuit-description) <br>
   7.2. [QASM-Based Compilation Path](#72-qasm-based-compilation-path) <br>
   7.3. [MLIR-Based Compilation Path](#73-mlir-based-compilation-path) <br>

8. [Limitations & Open Problems](#8-limitations--open-problems)

9. [Conclusion](#9-conclusion)

10. [Future Work](#10-future-work)

11. [Appendix](#11-appendix)


# 1. Abstract / Executive Summary

## 0. Abstract / Executive Summary

Fault-tolerant quantum computation introduces compilation challenges that extend beyond flat, gate-level circuit descriptions. As quantum systems scale, managing noise, error propagation, and the overhead of quantum error correction becomes central, requiring explicit representation of logical qubits, error-correcting codes, syndrome extraction, and classical feedback [3](#ref-gottesman), [4](#ref-kitaev). Existing intermediate representations such as QASM, while effective as low-level execution formats, lack the structural expressiveness needed to model these concepts and to support fault-tolerance–aware optimization [2](#ref-qasm).

This notebook explores the use of **Multi-Level Intermediate Representation (MLIR)** as a foundation for fault-tolerant quantum compilation and compares it with QASM-based workflows [1](#ref-mlir). By leveraging MLIR’s support for multiple abstraction levels, domain-specific dialects, and progressive lowering, fault tolerance can be elevated to a first-class compiler abstraction. This enables structured reasoning about error-correction overheads, scheduling constraints, and hardware-specific trade-offs. The objective is not to replace QASM, but to demonstrate why MLIR provides a more suitable internal representation for scalable, fault-tolerant quantum compilers.


# 2. Background & Motivation

## 2.1 Why Fault-Tolerant Compilation is Hard

Fault-tolerant quantum computation fundamentally alters the nature of what a compiler must reason about. In contrast to near-term quantum programs—where compilation primarily involves mapping an abstract circuit to a sequence of physical gates—fault-tolerant execution requires preserving and manipulating *encoded* quantum information across many physical qubits. This distinction gives rise to several structural challenges that cannot be addressed through gate substitution alone.

The most immediate challenge is the distinction between **physical and logical qubits**. A logical qubit is encoded across a *code block* consisting of many physical qubits, such that the logical state resides in a protected subspace of the larger Hilbert space [5](#ref-gottesman). Operations on logical qubits must therefore respect the structure of the underlying error-correcting code, including its stabilizers, distance, and decoding assumptions. Compilation must preserve this structure across all transformations, as violations can render error correction ineffective.

A second challenge arises from **error propagation through multi-qubit gates**. In fault-tolerant settings, it is not sufficient to consider whether a gate is functionally correct; one must also analyze how physical faults propagate across code blocks. Transversal gates are favored precisely because they limit error spread to at most one physical qubit per block, preserving correctability [6](#ref-eastin-knill). Non-transversal gates, which are unavoidable in universal computation, require carefully structured protocols such as magic-state injection and lattice surgery, introducing additional circuit regions with strict correctness constraints [7](#ref-bravyi-kitaev).

Fault tolerance further requires **explicit syndrome extraction and classical feedback loops**. Error correction is an active, ongoing process in which stabilizer measurements are repeatedly performed, decoded using classical algorithms, and conditionally applied corrections are scheduled based on measurement outcomes [8](#ref-fowler). These operations impose timing, synchronization, and data-dependency constraints that fundamentally exceed the expressiveness of flat circuit descriptions. As a result, fault-tolerant compilation must reason about *regions*, *cycles*, and *classical–quantum interaction*, not merely unitary transformations.

Together, these requirements motivate the central claim of this work:

> **Fault tolerance is not merely a matter of replacing gates, but of preserving and manipulating structure across multiple abstraction levels.**

This structural nature of fault tolerance directly motivates the need for intermediate representations that can explicitly encode hierarchy, semantics, and constraints—capabilities that traditional gate-level IRs such as QASM were not designed to provide.


## 2.2 What an Intermediate Representation Must Support for Fault Tolerance

Fault tolerance is “structure” because the unit of reasoning is no longer a flat gate list; it is a composition of **(i) encoded data layouts**, **(ii) repeated syndrome-extraction cycles**, **(iii) decoder-driven classical feedback**, and **(iv) hardware-constrained scheduling**. This is visible in practical demonstrations where progress comes from *end-to-end* co-design across code layout, measurement circuits, decoding, and control—not from gate-level expressivity alone [9](#ref-google-scaling), [10](#ref-google-below-threshold), [13](#ref-ibm-qldpc).

To make fault tolerance a first-class concern, an intermediate representation (IR) must support the following requirements. These become the evaluation rubric used later when comparing MLIR and QASM.


### R1. Multi-level abstraction (logical → encoded → syndrome → physical)

Fault-tolerant execution requires multiple “views” of the same program: a **logical circuit**, an **encoded layout** (e.g., surface-code patch or LDPC graph), the **syndrome extraction schedule**, and a **physical pulse/gate schedule**. Demonstrations of surface-code memories explicitly report performance “per error-correction cycle,” reflecting this layered nature rather than single-shot circuit execution [10](#ref-google-below-threshold). Likewise, LDPC-based memories are evaluated end-to-end as protocols that combine check measurements, decoding, and correction—not merely circuits [13](#ref-ibm-qldpc).

**Real-life implementation examples**

* **Google**: surface-code logical memories are executed as repeated rounds of stabilizer measurements with performance reported per cycle, and improvements are demonstrated by scaling the code distance (e.g., distance-5 vs distance-7) [10](#ref-google-below-threshold), building on earlier scaling studies [9](#ref-google-scaling).
* **IBM**: end-to-end fault-tolerant memory experiments using qLDPC (bivariate bicycle) codes emphasize the protocol level—checks, decoding, and corrections—as an integrated system [13](#ref-ibm-qldpc).

### R2. Explicit logical vs physical separation (code blocks, distance, layouts)

A compiler must explicitly represent **logical qubits** and their **encodings**, because correctness depends on **code distance, layout geometry, and which qubits are data vs measurement (ancilla)**. In practice, the “same algorithm” means very different physical executions depending on how the logical qubit is embedded and how checks are measured.

**Real-life implementation examples**

* **Google**: surface-code implementations separate **data qubits** from **measure qubits** and perform repeated stabilizer checks; scaling experiments explicitly compare different code distances and patch sizes [9](#ref-google-scaling), [10](#ref-google-below-threshold).
* **IBM**: hardware connectivity constraints (e.g., heavy-hex–style sparsity) influence which syndrome extraction circuits are practical, motivating structured circuits (e.g., flag-qubit methods) and explicit layout-aware compilation [14](#ref-ibm-flag), [15](#ref-ibm-surfacecode-benchmark).


### R3. Error model annotations (crosstalk, leakage, biased noise, correlated faults)

Fault tolerance depends on *how* errors occur (stochastic vs coherent, leakage, correlated events), and modern QEC evaluations increasingly incorporate hardware-realistic noise. A useful IR must attach error-relevant metadata at the right abstraction level (logical op, cycle, region), not treat all gates as identical.

**Real-life implementation examples**

* **Google**: decoding work trained and evaluated on real device data incorporates realistic noise signatures (including leakage/crosstalk information), demonstrating that decoding and error modeling are central to the system’s achieved logical error rates [11](#ref-google-decoder).
* **Google (scaling study)**: identifies rare damaging events and error floors that only appear when running many rounds and large distances—effects invisible in one-shot gate lists [9](#ref-google-scaling).

### R4. Scheduling & timing awareness (rounds, cycles, latency, feed-forward)

Fault-tolerant systems are inherently **time-structured**: stabilizers are measured in rounds; decoding and classical processing must complete within timing budgets; and corrections may be applied conditionally. The “program” includes a repeated schedule, not just a DAG of gates.

**Real-life implementation examples**

* **Google**: below-threshold demonstrations explicitly treat the memory as a per-cycle process; one variant integrates a **real-time decoder**, emphasizing latency and control-loop integration as part of the implementation [10](#ref-google-below-threshold).
* **IBM**: practical syndrome extraction experiments on real devices highlight circuit constructions and routing constraints tied to the hardware’s coupling graph—constraints that manifest as scheduling structure, not merely gate selection [14](#ref-ibm-flag).

### R5. Classical–quantum feedback as a first-class concept (decoders, conditionals, control flow)

Fault tolerance requires classical processing in the loop: syndrome bits are decoded to infer likely errors, and corrections or frame updates follow. This cannot be cleanly represented if the IR only models unitary quantum evolution. A compiler must represent decoder calls, measurement streams, and conditional behavior as structured entities.

**Real-life implementation examples**

* **Google**: decoder design and evaluation is treated as a first-class component of surface-code performance, including machine-learning decoders trained on hardware data [11](#ref-google-decoder).
* **IBM**: LDPC-based fault-tolerant memory protocols and their practical decoding requirements treat classical decoding and its feasibility as central to architecture decisions [13](#ref-ibm-qldpc).


### Implication for IR choice

These requirements show why fault tolerance is fundamentally **structural**: it is defined by *hierarchy (levels), repeated regions (cycles), dataflow (syndrome streams), and constraints (layout/timing)*. An IR that cannot represent these structures explicitly forces them into ad-hoc backend logic, making analysis, optimization, and portability significantly harder. This motivates the later comparison: QASM remains valuable as an execution format, but MLIR-style multi-level representations are a natural fit for encoding fault tolerance as a compiler-native concern.

# 3. QASM as an Intermediate Representation


## 3.1 What QASM Is Designed For

Below is **drop-in notebook content** for:

## 2. QASM as an Intermediate Representation

### 2.1 What QASM Is Designed For

…and it **dives into why QASM stays confined to “assembly-like” features** and **doesn’t grow into a fault-tolerant compiler IR**, with **concrete code examples**.

Reference numbering starts from **15**, using your **clickable anchor** style.

---

## 2. QASM as an Intermediate Representation

### 2.1 What QASM Is Designed For

OpenQASM was introduced as a **quantum assembly language** for describing **low-depth physical circuits** that can be executed on real devices, with a deliberately simple, textual model built around qubits, gates, measurement, reset, and limited classical control [15](#ref-openqasm2). OpenQASM 3 expands this model with richer classical computation, timing, and additional constructs to describe a broader set of experiments and control flows [16](#ref-openqasm3), [17](#ref-openqasm3spec). However, even with these additions, QASM’s center of gravity remains the same: it is best understood as an **execution-facing program format** rather than a **multi-level optimization IR**.

This design choice is beneficial for hardware execution, but it becomes a bottleneck for fault tolerance because fault tolerance is not merely “more gates”—it is repeated **structure**: code blocks, syndrome extraction rounds, decoding, feedback, and scheduling constraints. In real QEC demonstrations (e.g., surface-code memories), the unit of progress is *not* a single circuit but repeated rounds of stabilizer measurement integrated with decoding and control-loop constraints [18](#ref-google-below-threshold), [19](#ref-google-scaling). Representing that as a flat assembly stream loses the semantics the compiler needs to optimize.

---

### What QASM *naturally* supports (and why it’s still “assembly”)

#### Feature A — Flat gate lists over physical qubits

QASM excels at expressing *this*:

```qasm
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

h q[0];
cx q[0], q[1];
measure q[0];
```

This is exactly the “hardware-near” model described in OpenQASM’s original design goals [15](#ref-openqasm2).

#### Feature B — Measurement, reset, limited classical flow (expanded in OpenQASM 3)

OpenQASM 3 adds more classical expressiveness:

```qasm
OPENQASM 3;
qubit q;
bit c;

h q;
c = measure q;
if (c) x q;
```

This aligns with OpenQASM 3’s intent to support a broader set of circuits and control, including timing/pulse-facing extensions [16](#ref-openqasm3), [17](#ref-openqasm3spec).

So far so good.

---




## 3.2 Example: Logical Circuit in QASM
> comparison with MLIR

### Example 1: “One logical qubit” in QASM (conceptual mismatch)

#### Logical intent

> A **single logical qubit**, encoded using a surface code of distance 3.

#### QASM representation

```qasm
OPENQASM 3;

// Pretend this is one logical qubit
qubit data[9];     // 9 data qubits
qubit anc[8];      // 8 ancilla qubits
```

#### What QASM actually knows

* 17 unrelated physical qubits
* No idea:

  * that these qubits form **one code block**
  * what the **code distance** is
  * which stabilizers exist
  * what constitutes a valid logical state

💡 **Key point**:
The *logical qubit* exists only in the programmer’s head or external documentation.

---

### Example 2: Logical Hadamard gate (transversal, but implicit)

#### Logical intent

> Apply **H** to a logical qubit using transversal H on all data qubits.

#### QASM representation

```qasm
// Logical H (by convention)
for uint i in [0:8] {
  h data[i];
}
```

#### What QASM cannot express

* That this is a **single logical operation**
* That it is **transversal**
* That error propagation is bounded by design

To QASM, this is just **9 independent H gates**.

---

### Example 3: Logical CNOT between two logical qubits

#### Logical intent

> Logical CNOT between two surface-code patches.

#### QASM representation

```qasm
// Logical qubit A: dataA[0..8]
// Logical qubit B: dataB[0..8]

for uint i in [0:8] {
  cx dataA[i], dataB[i];
}
```

#### Why this is fragile

* Nothing prevents:

  * mixing indices
  * cross-block interactions
  * unsafe scheduling
* Compiler cannot verify:

  * “one error per block”
  * block alignment
  * code compatibility

💡 **This works only because the programmer is disciplined.**

---

### Example 4: Syndrome extraction (core FT structure lost)

#### Logical intent

> One **round of stabilizer measurement** for a surface code.

### QASM representation

```qasm
OPENQASM 3;
bit s[4];

// Entangle ancilla with data (Z-stabilizer example)
cx data[0], anc[0];
cx data[1], anc[0];
cx data[2], anc[0];
cx data[3], anc[0];

// Measure stabilizer
s[0] = measure anc[0];
reset anc[0];
```

### What is missing

* No notion of:

  * “this is a stabilizer”
  * “this measurement is a detector”
  * “this belongs to cycle t”
* No boundary marking:

  * where decoding happens
  * where corrections apply

To QASM, this is just **entanglement + measurement**.

---

### Example 5: Repeated QEC cycles (structure flattened)

#### Logical intent

> Run **100 rounds of error correction**.

#### QASM representation

```qasm
for uint round in [0:99] {
  // stabilizer measurements
  // syndrome bits stored
}
```

#### Why this is insufficient

* The compiler cannot know:

  * each loop iteration is a **QEC cycle**
  * measurements are **temporally correlated**
  * decoder must run **between iterations**
* Cannot optimize across rounds
* Cannot reason about **logical error rate per cycle**

💡 **Cycles are syntax, not semantics.**

---

### Example 6: Logical T gate (magic state injection hidden)

#### Logical intent

> Apply a logical **T gate**, requiring magic state injection.

#### QASM representation (simplified)

```qasm
// Prepare magic state (by convention)
h anc;
t anc;

// Entangle with data
cx anc, data[0];

// Measure ancilla
bit m;
m = measure anc;

// Conditional correction
if (m) z data[0];
```

#### What QASM cannot represent

* That:

  * this is **one logical T**
  * ancilla is a **magic state**
  * distillation was required
* No way to:

  * cost T gates
  * batch distillation
  * schedule factories

---

### Summary Table 

| Logical concept  | Can QASM express it? | How                     |
| ---------------- | -------------------- | ----------------------- |
| Logical qubit    | ❌                    | Naming convention only  |
| Code distance    | ❌                    | External assumption     |
| Logical gate     | ❌                    | Expanded physical gates |
| Syndrome cycle   | ❌                    | Loop without semantics  |
| Decoder boundary | ❌                    | Not representable       |
| Pauli frame      | ❌                    | External logic          |
| FT scheduling    | ❌                    | Backend heuristics      |

---

> **QASM can describe fault-tolerant circuits, but it cannot encode fault tolerance as structure—only as convention.**




> ⚠️ Note: This is **conceptual MLIR**, not tied to an existing upstream quantum dialect. That’s intentional—you are proposing structure, not an API.

---

### Example 1: “One logical qubit”

#### QASM (recap)

```qasm
qubit data[9];
qubit anc[8];
```

##### Problem

* No notion of *one logical qubit*
* No code identity, distance, or invariants

---

#### MLIR version (logical allocation)

```mlir
%L0 = quantum.logical.alloc
  { code = "surface",
    distance = 3,
    data_qubits = 9,
    ancilla_qubits = 8 }
```

##### MLIR features used

* **Strong typing** (`logical` qubit vs physical)
* **Attributes with semantics**
* **Single SSA value represents a code block**

##### Why this matters for fault tolerance

* Compiler *knows*:

  * this is **one fault-tolerance unit**
  * how many errors are correctable
* Enables:

  * block-aware scheduling
  * safe transversal checks
  * logical error-rate estimation

---

### Example 2: Logical Hadamard (transversal gate)

#### QASM (recap)

```qasm
for uint i in [0:8] {
  h data[i];
}
```

##### Problem

* Compiler sees **9 unrelated H gates**
* Cannot verify transversality

---

#### MLIR version

```mlir
quantum.logical.h %L0
  { transversal = true }
```

##### MLIR features used

* **Operation-level semantics**
* **Transversality annotation**
* **Single logical op → many physical ops**

##### Why this matters for fault tolerance

* Compiler can:

  * prove bounded error propagation
  * parallelize safely
  * avoid unnecessary EC cycles

---

### Example 3: Logical CNOT between two logical qubits

#### QASM (recap)

```qasm
for uint i in [0:8] {
  cx dataA[i], dataB[i];
}
```

##### Problem

* No guarantee of block alignment
* Unsafe interactions possible

---

#### MLIR version

```mlir
quantum.logical.cnot %L0, %L1
  { transversal = true,
    alignment = "pairwise" }
```

##### MLIR features used

* **Block-aware operands**
* **Explicit interaction constraints**
* **Verifier-enforced safety**

##### Why this matters for fault tolerance

* Compiler enforces:

  * ≤1 error per block
  * compatible code parameters
* Prevents illegal cross-block coupling

---

### Example 4: Syndrome extraction (single round)

#### QASM (recap)

```qasm
cx data[0], anc[0];
...
s[0] = measure anc[0];
```

##### Problem

* Syndrome semantics are implicit
* Decoder boundary invisible

---

#### MLIR version

```mlir
quantum.syndrome.round %L0 {
  %syn = quantum.syndrome.measure %L0
  quantum.decoder.invoke %syn
}
```

##### MLIR features used

* **Region-based structure**
* **Explicit syndrome object**
* **Decoder as first-class op**

##### Why this matters for fault tolerance

* Compiler understands:

  * measurement → decoding → correction flow
* Enables:

  * decoder scheduling
  * round fusion / reordering
  * hardware-specific decoding strategies

---

### Example 5: Repeated QEC cycles

#### QASM (recap)

```qasm
for uint round in [0:99] {
  // stabilizers
}
```

##### Problem

* Loop has no semantic meaning
* Cannot reason across rounds

---

#### MLIR version

```mlir
quantum.syndrome.cycle %L0
  { rounds = 100,
    cycle_time_ns = 800 } {
  quantum.syndrome.round %L0 { ... }
}
```

##### MLIR features used

* **Hierarchical regions**
* **Temporal attributes**
* **Cycle-level abstraction**

##### Why this matters for fault tolerance

* Compiler can:

  * compute logical error per cycle
  * co-schedule decoding
  * adapt distance dynamically

---

### Example 6: Logical T gate (magic state injection)

#### QASM (recap)

```qasm
h anc;
t anc;
cx anc, data[0];
m = measure anc;
if (m) z data[0];
```

##### Problem

* Magic state semantics lost
* Cost invisible to compiler

---

#### MLIR version

```mlir
quantum.logical.t %L0
  { requires_magic = true,
    magic_type = "T",
    injection_protocol = "teleportation" }
```

##### MLIR features used

* **Non-transversal gate classification**
* **Resource annotation**
* **Protocol-level abstraction**

##### Why this matters for fault tolerance

* Compiler can:

  * batch T gates
  * schedule distillation factories
  * trade space vs time
* Essential for realistic FT cost modeling

---

### Summary Comparison Table (use this verbatim)

| Aspect          | QASM         | MLIR (FT-aware)      |
| --------------- | ------------ | -------------------- |
| Logical qubits  | ❌ Convention | ✅ Typed              |
| Code parameters | ❌ External   | ✅ Attributes         |
| Transversality  | ❌ Implicit   | ✅ Verifiable         |
| Syndrome cycles | ❌ Flat loops | ✅ Structured regions |
| Decoding        | ❌ External   | ✅ First-class        |
| Magic states    | ❌ Hidden     | ✅ Explicit           |
| FT optimization | ❌ Heuristic  | ✅ Systematic         |

---

#### One sentence that **ties the section together**

> **MLIR allows fault tolerance to be represented as compiler-visible structure, whereas QASM confines it to programmer discipline and backend assumptions.**


## 3.3 Attempting Fault Tolerance in QASM


### Where QASM *stops growing* for fault tolerance

Fault-tolerant compilation needs the IR to represent concepts that are **not** “just more gates.” Below are the core failure modes you can use in your comparison.


### (1) No first-class notion of *logical qubits* / code blocks / code distance

In a fault-tolerant system, you want to express:

* “This is **one logical qubit**”
* encoded using (say) surface code
* at distance **d = 7**
* with a defined measurement schedule

In QASM, you can only allocate *physical qubits*:

```qasm
qubit data[49];     // (pretend these are a surface-code patch)
qubit anc[24];      // (pretend these are measurement qubits)
```

But this is just a naming convention. There is nothing in the language that tells the compiler:

* which subset constitutes *one logical qubit*
* what the code distance is
* what the stabilizers are
* what “a cycle” means

So any FT semantics live outside the IR—in scripts, backend assumptions, or compiler code.

---

### (2) QEC is fundamentally *cyclic structure*, but QASM makes it “just a loop”

A surface-code memory is characterized by repeated syndrome rounds, and performance is discussed **per round / per cycle** in real experiments [18](#ref-google-below-threshold), [19](#ref-google-scaling). In QASM, the best you can do is write a loop, but the IR does not understand that the body is “one syndrome cycle” with invariants.

```qasm
OPENQASM 3;
qubit data[49];
qubit anc[24];
bit s[24];

for uint r in [0:999] {
  // Entangle ancillas with data (stabilizer measurement pattern)
  // ... dozens/hundreds of gates here ...

  // Measure ancillas (syndrome)
  for uint i in [0:23] {
    s[i] = measure anc[i];
    reset anc[i];
  }

  // Where is decoding represented?
  // Where is the correction frame updated?
}
```

This is the crux:

* QASM can **spell** the loop,
* but it cannot **represent the semantics** that compilers need:

  * “this region is a syndrome-extraction round”
  * “these measurements form a detector graph”
  * “this is a decoder boundary”
  * “corrections are Pauli-frame updates, not physical X/Z gates”

---

### (3) No first-class representation for decoding / detector events / Pauli frame

In practice, surface-code fault tolerance relies on:

* transforming measurement streams into **detector events**
* running a **decoder**
* updating a **Pauli frame**
* possibly doing time-critical feedback [18](#ref-google-below-threshold)

In QASM, you can store measurement bits:

```qasm
bit s[24];
s[i] = measure anc[i];
```

…but the semantics of those bits (“detectors”, “syndrome graph edges”, “decoder input”, “frame update”) are not representable. You are forced into one of two bad options:

1. **Inline fake correction gates** (changes error propagation and schedule)
2. Push decoding into an external system with no IR-level hooks

Either way, the compiler loses the ability to optimize fault tolerance as structure.

---

### (4) “More features” in QASM ≠ “FT-ready IR”

OpenQASM 3 explicitly acknowledges the need to describe circuits beyond simple gate lists (including timing and control) [16](#ref-openqasm3), [17](#ref-openqasm3spec). That is an important evolution—but it is still not the same as:

* a multi-level IR where *logical intent* survives lowering
* an IR where “fault-tolerant regions” are analyzable entities
* an IR where code parameters (distance, layout) are first-class

So QASM grows “deeper” as an execution language, but it does not naturally become a **compiler-native FT IR**.

### Summary Claim

> **QASM can describe fault-tolerant circuits, but it cannot *reason about fault tolerance* as structure.**
> As a result, key FT decisions become external conventions or backend heuristics, which limits optimization, portability, and correctness auditing.

This is precisely why a multi-level representation (e.g., MLIR dialect stack) becomes attractive: the compiler can preserve *structure* explicitly instead of reconstructing it from a flat instruction stream.


# 4. MLIR: A Compiler-Native Intermediate Representation

## 4.1 What MLIR Is (Conceptual Overview)

Multi-Level Intermediate Representation (MLIR) is a compiler infrastructure designed around the idea that **program meaning must be preserved across multiple abstraction levels**, rather than collapsed prematurely into a single flat representation [20](#ref-mlir). Instead of enforcing one universal IR, MLIR allows domain-specific abstractions to coexist through **dialects**, each capturing semantics relevant at a particular level of reasoning.

This design aligns naturally with fault-tolerant quantum computation, where correctness depends on preserving *structure*—logical qubits, encoded layouts, syndrome cycles, and decoding boundaries—across compilation stages. In contrast to assembly-style IRs, MLIR enables these structures to remain explicit until they are intentionally lowered.

At a conceptual level, MLIR provides four features that are directly relevant to fault tolerance:

---

### Multi-level abstraction

MLIR supports multiple representations of the same program, connected through explicit lowering steps. This mirrors the structure of fault-tolerant execution, which transitions from **logical circuits** to **encoded protocols**, to **syndrome extraction schedules**, and finally to **physical operations**. Preserving these levels is essential for reasoning about logical error rates and code distance, as emphasized in modern QEC demonstrations [21](#ref-fowler-surface), [22](#ref-google-scaling).

---

### Dialects as semantic boundaries

Dialects allow fault-tolerance–specific concepts (e.g., logical qubits, transversal gates, magic-state injection) to be represented explicitly rather than inferred from gate patterns. This reflects how real systems separate concerns: surface-code experiments, for example, distinguish between data qubits, measurement qubits, and decoding stages as first-class components of the protocol [22](#ref-google-scaling), [23](#ref-google-below-threshold).

---

### Progressive lowering

Rather than committing early to hardware-specific details, MLIR encourages delaying irreversible decisions. This is critical for fault tolerance, where choices such as code distance, decoding strategy, or magic-state scheduling depend on hardware error rates and timing constraints that are often only known late in the compilation pipeline [24](#ref-gidney-ftcosts). Progressive lowering allows these decisions to be made when sufficient information is available.

---

### Strong typing and semantic attributes

MLIR operations and values can carry rich semantic information through types and attributes. For fault tolerance, this enables properties such as *transversality*, *error-spreading behavior*, *code parameters*, and *resource cost* to be explicitly attached to operations. This mirrors how fault-tolerant protocols are analyzed in the literature—not just by gate count, but by structured properties such as error propagation and threshold behavior [25](#ref-eastin-knill).

---

### Core implication

Together, these features allow MLIR to treat fault tolerance not as an emergent property of long gate sequences, but as **compiler-visible structure**. This directly contrasts with flat IRs, where the compiler must reconstruct structure from patterns or rely on external conventions. As fault-tolerant systems grow in scale and complexity, preserving this structure becomes essential for correctness, optimization, and portability.


## 4.2 Why MLIR Is a Better Fit for Fault Tolerance

Fault tolerance is *structure*: logical qubits encoded into code blocks, repeated syndrome-extraction **cycles**, decoder-in-the-loop **classical feedback**, and hardware-driven scheduling constraints. This is exactly how leading demonstrations report results—e.g., surface-code memories evaluated over many rounds and across code distances, with decoding integrated as part of the system behavior [29](#ref-google-below-threshold), [30](#ref-google-scaling). ([Nature][1])

QASM (even OpenQASM 3) is primarily an **execution-facing assembly**: it can *describe* these protocols as long gate/measurement programs, but it does not preserve fault-tolerance structure as compiler-visible entities [27](#ref-openqasm3), [28](#ref-openqasm2). MLIR, by design, is built to keep high-level semantics available across lowering stages via dialects, regions, and attributes [26](#ref-mlir). ([arXiv][2])

### Mapping FT requirements → MLIR features

| FT requirement (what compilers must reason about)                    | MLIR feature (what makes it representable)           |
| -------------------------------------------------------------------- | ---------------------------------------------------- |
| **Logical abstraction** (logical qubits, code blocks, distance)      | **Dialects + types** (e.g., `quantum.logical` types) |
| **Multiple views** (logical → encoded → syndrome → physical)         | **Multi-level IR + progressive lowering**            |
| **Error models / FT metadata** (transversal, error-spreading, costs) | **Attributes** on ops/regions                        |
| **Scheduling structure** (cycles/rounds, timing boundaries)          | **Regions + structured control flow**                |
| **Adaptivity** (late binding: distance, decoder choice, layout)      | **Rewrite passes + verifiers**                       |

### Minimal examples showing *structure* (not just syntax)

**Logical abstraction (typed logical qubit + code parameters):**

```mlir
%L = quantum.logical.alloc { code="surface", distance=7 }
```

Needed for FT: makes “one logical qubit / one code block” explicit (not a naming convention).

**Syndrome cycle as a first-class region (so the compiler can optimize cycles):**

```mlir
quantum.syndrome.cycle %L { rounds = 100 } {
  %syn = quantum.syndrome.measure %L
  quantum.decoder.invoke %syn
}
```

Needed for FT: preserves *round boundaries* + *decoder boundary* as analyzable structure (critical in real surface-code implementations). ([Nature][1])

**Non-transversal gate tagged for magic-state resources (so T-cost is visible):**

```mlir
quantum.logical.t %L { requires_magic=true, magic_type="T" }
```

Needed for FT: enables compiler-level scheduling/batching of magic-state factories instead of hiding them inside flattened circuits.

### Why QASM “confines itself”

OpenQASM 2/3 intentionally centers on qubits, gates, measurement, and classical flow for describing executable programs [27](#ref-openqasm3), [28](#ref-openqasm2). That’s excellent for interoperability and execution, but it means fault-tolerance semantics (code distance, syndrome rounds, decoder interfaces, Pauli frame) live outside the IR—typically in backend code, conventions, or external tooling—making FT optimization and verification harder than if the structure were explicit. ([arXiv][3])


# 5. Designing a Fault-Tolerant Quantum MLIR Stack

## 5.1 Dialect Stack Overview

Fault-tolerant quantum computation is inherently hierarchical: correctness emerges not from individual gates, but from the coordinated interaction of **logical encodings**, **error-correction protocols**, **repeated syndrome cycles**, and **hardware-constrained execution**. Modern demonstrations of fault tolerance—particularly surface-code–based systems—are evaluated and reasoned about at these distinct layers, rather than as monolithic circuits [33](#ref-google-scaling), [34](#ref-google-below-threshold). This layered reality motivates a compiler architecture in which fault tolerance is represented explicitly as **structure**, not reconstructed from flat gate sequences.

We propose a multi-dialect MLIR stack that mirrors this structure:

``` cmd
quantum.logical
      ↓
quantum.encoded
      ↓
quantum.syndrome
      ↓
quantum.physical
```

Each dialect captures a **stable semantic boundary**, preserving information that is essential for fault-tolerant reasoning while intentionally abstracting away lower-level details until they are required.

---

### `quantum.logical` — *What the algorithm means*

This dialect represents **logical intent**: logical qubits, logical gates, and algorithmic structure, independent of how error correction is implemented.

**Knows**

* Logical qubits and registers
* High-level operations (H, CNOT, T, etc.)
* Algorithmic control flow

**Does not know**

* Which error-correcting code is used
* Code distance or layout
* Syndrome extraction or decoding

This separation reflects how algorithms are specified independently of hardware and QEC details in theory [31](#ref-gottesman-qec).

---

### `quantum.encoded` — *How logical information is protected*

This dialect introduces **encoding structure**: code blocks, code parameters (e.g., distance), and gate properties such as transversality or the need for magic states.

**Knows**

* Error-correcting code (e.g., surface code, LDPC)
* Code distance and block identity
* Whether an operation is transversal or non-transversal

**Does not know**

* Exact stabilizer circuits
* Measurement scheduling
* Physical gate implementations

This corresponds to how fault-tolerant protocols are analyzed at the encoded level, where properties like error propagation and threshold behavior are determined [32](#ref-eastin-knill), [35](#ref-bravyi-kitaev).

---

### `quantum.syndrome` — *How errors are detected and interpreted*

This dialect makes **error correction explicit** by representing syndrome extraction rounds, detector events, and decoder invocation as first-class structure.

**Knows**

* Stabilizer measurement structure
* Repeated QEC cycles / rounds
* Decoder boundaries and classical feedback

**Does not know**

* Pulse-level timing
* Hardware-native gate sets

This layer reflects real implementations, where performance is measured per round and decoding is an integral part of the protocol, not an afterthought [33](#ref-google-scaling), [34](#ref-google-below-threshold).

---

### `quantum.physical` — *How the hardware executes*

This dialect captures the **hardware-facing reality**: native gates, connectivity, timing constraints, and control instructions.

**Knows**

* Physical qubits and couplings
* Native gates (e.g., CX, MS, iSWAP)
* Scheduling and latency constraints

**Does not know**

* Logical semantics
* Error-correction intent

This matches the execution model targeted by assembly-level languages such as QASM, which are well-suited for this lowest layer but not for higher-level fault-tolerant reasoning [36](#ref-openqasm3).

---

### Core insight

Each dialect preserves exactly the information needed at its level—and no more. Together, they allow the compiler to **carry fault-tolerance structure forward**, instead of flattening it prematurely. This directly reflects how fault-tolerant quantum computing is practiced and evaluated in the literature: as a stack of interacting protocols rather than a single expanded circuit.




## 5.2 Logical Dialect

The **logical dialect** captures *algorithmic intent* independent of error correction. At this level, the compiler reasons about **logical qubits**, **logical operations**, and **control flow**, without committing to how information is protected or executed physically. This mirrors the theoretical separation between algorithms and encoding in quantum error correction literature [37](#ref-gottesman-qec).

### Logical qubits and code blocks

A logical qubit represents encoded quantum information abstractly. While a *code block* will later correspond to many physical qubits, the logical dialect intentionally hides that structure to preserve portability and clarity. This separation is foundational in stabilizer-based models of quantum computation [37](#ref-gottesman-qec).

### Gate semantics (H, CNOT, T)

Logical gates express **what transformation is intended**, not *how* it is implemented fault-tolerantly. Whether a gate is transversal or requires ancillary protocols is deferred to lower dialects.

### Example

```mlir
%q = quantum.logical.alloc { code = "surface", distance = 7 }
quantum.logical.h %q
```

### Why this matters for FT

* Preserves **algorithmic meaning** across compilation
* Enables late binding of encoding choices
* Prevents premature flattening into physical gates

---

## 5.3 Encoded / Fault-Tolerant Dialect

The **encoded (FT) dialect** introduces the structure required to reason about **error propagation and protection**. At this level, the compiler becomes aware of **code distance**, **transversality**, and **non-transversal resource requirements**, reflecting how fault tolerance is analyzed in practice [38](#ref-eastin-knill), [39](#ref-bravyi-kitaev).

### Transversal gates

Transversal gates are explicitly marked because they bound error spread to one physical qubit per code block—an essential fault-tolerance criterion [38](#ref-eastin-knill).

### Magic state requirements

Universal computation requires non-transversal gates (e.g., T), which must be implemented via **magic-state injection and distillation**, a dominant cost driver in FT architectures [39](#ref-bravyi-kitaev), [40](#ref-gidney-ftcosts).

### Code distance

The encoded dialect makes **code distance** explicit, allowing the compiler to trade space, time, and logical error rates.

### Example

```mlir
quantum.encoded.t %q
  { requires_magic = true, magic_type = "T" }
```

### Why this matters for FT

* Makes **error-spreading behavior visible**
* Enables **FT-aware cost modeling**
* Allows scheduling and batching of magic states

---

## 5.4 Syndrome & Correction Dialect

Fault tolerance is operationally defined by **repeated syndrome extraction, decoding, and correction**. The syndrome dialect makes this *cyclic structure* explicit, matching how real systems are evaluated experimentally [41](#ref-fowler-surface), [42](#ref-google-scaling).

### Syndrome extraction regions

Syndrome rounds are represented as structured regions, not implicit gate sequences. This allows the compiler to reason about **round boundaries** and **temporal correlations**.

### Classical decoding hooks

Decoding is a first-class operation: measurement results are consumed by a decoder, which informs correction or Pauli-frame updates. This reflects real implementations where decoding is inseparable from QEC performance [42](#ref-google-scaling), [43](#ref-google-below-threshold).

### Timing constraints

Syndrome cycles impose strict latency and synchronization constraints that must be preserved across compilation.

### Example

```mlir
%syndrome = quantum.syndrome.measure %block
quantum.decoder.call %syndrome
```

### Why this matters for FT

* Preserves **cycle-level structure**
* Enables **decoder-aware scheduling**
* Supports realistic performance estimation (per round)

---

## 5.5 Physical Dialect

The **physical dialect** represents how instructions are executed on real hardware. This layer corresponds closely to execution-level languages such as QASM and hardware control systems [44](#ref-openqasm3).

### Hardware-native gates

Operations are expressed in the device’s native gate set (e.g., CX, MS, iSWAP), reflecting platform-specific constraints.

### Scheduling

The physical dialect makes timing, parallelism, and control dependencies explicit—critical for meeting coherence and feedback deadlines.

### Connectivity constraints

Limited qubit connectivity is enforced here, influencing routing and scheduling but no longer affecting logical correctness.

### Why this matters for FT

* Anchors abstract FT structure to real devices
* Ensures correctness under hardware constraints
* Provides a clean boundary between compilation and execution

> **Each dialect preserves exactly the structure needed for fault tolerance at its level—preventing the premature flattening that undermines FT reasoning in gate-only IRs.**


# 6. Comparative Analysis: MLIR vs QASM


Fault-tolerant quantum computation exposes a fundamental divergence between **execution languages** and **compiler intermediate representations**. While QASM is intentionally designed as a hardware-facing assembly language, fault tolerance requires the compiler to preserve *semantic structure* across abstraction levels—a requirement that MLIR is explicitly designed to meet [45](#ref-mlir), [46](#ref-openqasm3).

---

## 6.1 Expressiveness

| Feature                | QASM | MLIR |
| ---------------------- | ---- | ---- |
| Logical qubits         | ❌    | ✅    |
| Code distance          | ❌    | ✅    |
| Error models           | ❌    | ✅    |
| Fault-tolerance intent | ❌    | ✅    |

**Explanation**

QASM represents programs as sequences of operations on *physical qubits*, optionally enriched with classical control. Even in OpenQASM 3, there is no notion of a logical qubit, code block, or encoding invariant—these exist only as external conventions or backend assumptions [46](#ref-openqasm3).

In contrast, MLIR allows these concepts to be represented explicitly via **dialects, types, and attributes**, enabling the compiler to distinguish between logical intent, encoded structure, and physical execution [45](#ref-mlir). This distinction mirrors how fault-tolerant systems are reasoned about in practice, where correctness is defined at the level of encoded information rather than individual gates [47](#ref-gottesman-qec).

---

## 6.2 Optimizations Possible

Fault tolerance introduces optimization opportunities that are **structural**, not syntactic. These optimizations rely on explicit knowledge of code parameters, cycles, and resource constraints.

### Adaptive code distance

Real systems adjust protection strength based on error rates and workload requirements. Logical error suppression is demonstrated by increasing code distance rather than modifying gate sequences [48](#ref-google-scaling). In QASM, code distance is not representable; in MLIR, it can be an attribute subject to optimization passes.

### Error-correction cycle coalescing

Repeated syndrome extraction rounds dominate runtime in surface-code systems, and their scheduling critically impacts performance [49](#ref-fowler-surface). MLIR can represent these rounds as structured regions and optimize across them. In QASM, they appear as opaque loops with no semantic identity.

### Magic-state batching

Non-transversal gates (e.g., T) impose overwhelming overhead due to magic-state distillation, which must be globally scheduled and amortized [50](#ref-gidney-ftcosts). QASM hides this cost inside expanded circuits, whereas MLIR can expose it as a first-class resource, enabling compiler-level batching and trade-off analysis.

> **These optimizations are effectively impossible in QASM, but natural in MLIR.**

---

### Core takeaway

QASM can *express* fault-tolerant circuits, but it cannot *optimize* or *verify* fault tolerance because the relevant structure is not represented in the IR. MLIR, by preserving hierarchy and semantics across lowering stages, allows fault tolerance to be treated as a compiler-visible property rather than a backend convention.

> **QASM flattens fault tolerance into gates; MLIR preserves it as structure.**


## 6.3 Compiler Passes Enabled

Fault-tolerant quantum compilation requires compiler passes that operate on **semantic structure**, not just gate sequences. The passes listed below are routinely *implicit* in successful fault-tolerant implementations, but can only be made **explicit, analyzable, and optimizable** when the IR preserves fault-tolerance structure.

### FT-aware scheduling

Fault-tolerant execution is dominated by **error-correction cycles**, decoder latency, and feedback deadlines. Scheduling decisions must respect boundaries between syndrome rounds and ensure that classical decoding completes within the coherence window of the quantum state. In surface-code systems, performance is reported per round, not per circuit, highlighting the centrality of cycle-aware scheduling [53](#ref-google-below-threshold), [54](#ref-fowler-surface).

* **QASM**: Represents cycles as opaque loops; the compiler cannot distinguish stabilizer rounds from arbitrary repetition.
* **MLIR**: Represents cycles as structured regions, enabling passes that reorder, pipeline, or coalesce syndrome rounds without violating fault-tolerance invariants [51](#ref-mlir).

---

### Noise-aware lowering

Fault-tolerant compilation must adapt to **hardware-specific noise characteristics**, including biased noise, correlated faults, and leakage. Modern experiments show that logical error rates depend strongly on how decoding and scheduling are matched to the device’s actual noise profile [52](#ref-gottesman-qec), [55](#ref-google-scaling).

* **QASM**: Treats gates as uniform operations; noise models live outside the IR and cannot guide transformations.
* **MLIR**: Allows error-related metadata (e.g., error-spreading behavior, gate fidelity, leakage sensitivity) to be attached as attributes, enabling lowering passes that choose encodings, layouts, or protocols based on measured noise [51](#ref-mlir).

---

### Hardware co-design

Fault tolerance is inseparable from hardware architecture. Connectivity graphs, native gate sets, and measurement capabilities directly shape which error-correction protocols are viable. This is evident in platform-specific implementations of surface codes and LDPC-based memories, where layout and scheduling decisions are co-designed with hardware constraints [56](#ref-ibm-qldpc), [55](#ref-google-scaling).

* **QASM**: Hardware constraints are enforced late, often via routing or rejection, after FT structure has already been flattened.
* **MLIR**: Supports progressive lowering, allowing hardware constraints to influence earlier decisions such as code distance, stabilizer ordering, and decoder placement—without destroying logical semantics [51](#ref-mlir).

---

### Key insight

> **These compiler passes are not “optional optimizations”; they are required for correctness and scalability in fault-tolerant systems.**

QASM cannot naturally support them because the necessary structure—logical qubits, cycles, decoding boundaries, and error models—is not represented in the IR. MLIR enables these passes precisely because it preserves fault tolerance as *explicit structure* throughout compilation.

> **MLIR doesn’t just enable better compilation—it enables the *right* compiler passes for fault-tolerant quantum computing.**


# 7. Case Study: Fault-Tolerant Compilation Workflow

## 7.1 Logical Circuit Description

## 7.2 QASM-Based Compilation Path

## 7.3 MLIR-Based Compilation Path

# 8. Limitations & Open Problems

While MLIR provides a compelling foundation for representing fault tolerance as compiler-visible structure, its adoption for fault-tolerant quantum compilation is not without limitations. A realistic assessment of these challenges is essential for both practical deployment and future research.

---

### 8.1 MLIR Complexity

MLIR is a powerful but **non-trivial compiler infrastructure**. Designing dialects, verifiers, and lowering pipelines requires significant compiler engineering expertise. For quantum compilation, this complexity is amplified by the need to reason simultaneously about quantum semantics, classical control, timing, and hardware constraints. While MLIR was explicitly designed to manage such complexity through modularity and abstraction, the upfront engineering cost is substantially higher than that of flat, assembly-style representations [57](#ref-mlir).

**Implication:**
MLIR is best suited for compiler teams and long-lived toolchains, rather than lightweight or ad hoc quantum programming environments.

---

### 8.2 Steep Learning Curve

Effective use of MLIR requires familiarity with:

* SSA-based IR design
* Dialect and type system design
* Rewrite rules and verification passes

For quantum researchers and algorithm developers—many of whom are not compiler specialists—this learning curve can be a barrier. In contrast, QASM’s simplicity and textual nature make it more immediately accessible, even if that simplicity limits its long-term scalability [58](#ref-openqasm3).

**Implication:**
There is a need for higher-level tooling and documentation to bridge the gap between quantum domain expertise and MLIR-based compiler development.

---

### 8.3 Tooling Maturity for Quantum Workloads

Although MLIR itself is mature and widely used in classical domains, **quantum-specific MLIR ecosystems are still emerging**. Unlike classical domains (e.g., ML, linear algebra), there is no widely adopted, production-grade set of quantum MLIR dialects with stable semantics for fault tolerance.

Current quantum compiler stacks largely rely on custom IRs or extend QASM-like representations, reflecting the field’s early stage rather than a settled consensus [59](#ref-lao-quantum-ir).

**Implication:**
Early MLIR-based quantum compilers must invest in infrastructure that is not yet standardized, increasing development risk but also offering research opportunities.

---

### 8.4 Lack of Standardized Fault-Tolerant Quantum Dialects

Perhaps the most significant open problem is the **absence of a standard fault-tolerant quantum IR**. While the theoretical foundations of fault tolerance are well established, there is no agreement on:

* How to represent code blocks and distance in IR
* How to encode syndrome cycles and decoder interfaces
* How to model Pauli frames and classical–quantum interaction

As a result, any MLIR-based FT design today is necessarily *exploratory*. This mirrors the historical development of classical compiler IRs, where domain-specific abstractions only stabilized after sustained experimentation and community convergence [57](#ref-mlir), [60](#ref-quantum-compiler-survey).

**Implication:**
The lack of standardization is both a limitation and an opportunity: MLIR provides the flexibility to experiment, but widespread adoption will require community-driven convergence.

---

### Summary

These limitations do not negate the suitability of MLIR for fault-tolerant quantum compilation. Rather, they highlight that MLIR is a **long-term architectural choice**, aligned with the structural complexity of fault tolerance but demanding careful engineering, tooling investment, and ecosystem development.


> **Fault tolerance demands structural representations; MLIR provides them—but turning that potential into a standard remains an open, community-scale problem.**



# 9. Conclusion

# 10. Future Work

# 11. Appendix

## Refereces


<a id="ref-mlir"></a>
**[1]** [MLIR: A Compiler Infrastructure for the End of Moore’s Law](https://mlir.llvm.org/)

<a id="ref-qasm"></a>
**[2]** [OpenQASM 3.0 Specification](https://arxiv.org/abs/2104.14722)

<a id="ref-gottesman"></a>
**[3]** D. Gottesman, *Stabilizer Codes and Quantum Error Correction*, PhD Thesis, California Institute of Technology, 1997

<a id="ref-kitaev"></a>
**[4]** A. Kitaev, “Fault-tolerant quantum computation by anyons,” *Annals of Physics*, 2003

<a id="ref-gottesman"></a>
**[5]** D. Gottesman, *Stabilizer Codes and Quantum Error Correction*, PhD Thesis, California Institute of Technology, 1997.

<a id="ref-eastin-knill"></a>
**[6]** B. Eastin and E. Knill, “Restrictions on Transversal Encoded Quantum Gate Sets,” *Physical Review Letters*, vol. 102, no. 11, 2009.

<a id="ref-bravyi-kitaev"></a>
**[7]** S. Bravyi and A. Kitaev, “Universal quantum computation with ideal Clifford gates and noisy ancillas,” *Physical Review A*, vol. 71, 2005.

<a id="ref-fowler"></a>
**[8]** A. G. Fowler, M. Mariantoni, J. M. Martinis, and A. N. Cleland, “Surface codes: Towards practical large-scale quantum computation,” *Physical Review A*, vol. 86, 2012.


<a id="ref-google-scaling"></a>
**[9]** Google Quantum AI, “Suppressing quantum errors by scaling a surface code logical qubit,” *Nature*, 2023.

<a id="ref-google-below-threshold"></a>
**[10]** Google Quantum AI, “Quantum error correction below the surface code threshold,” *Nature*, 2025.

<a id="ref-google-decoder"></a>
**[11]** J. Bausch *et al.*, “Learning high-accuracy error decoding for quantum processors,” *Nature*, 2024.

<a id="ref-ibm-qldpc"></a>
**[13]** S. Bravyi *et al.*, “High-threshold and low-overhead fault-tolerant quantum memory,” *Nature*, 2024.

<a id="ref-ibm-flag"></a>
**[14]** Y. Kim *et al.*, “Effectiveness of the syndrome extraction circuit with flag qubits on IBM quantum computers,” *Quantum*, 2025.

<a id="ref-ibm-surfacecode-benchmark"></a>
**[15]** I. Hesner *et al.*, “Using detector likelihood for benchmarking quantum error correction,” *Physical Review A*, 2025.


<a id="ref-openqasm2"></a>
**[15]** A. W. Cross, L. S. Bishop, J. A. Smolin, and J. M. Gambetta, “Open Quantum Assembly Language,” arXiv:1707.03429 (2017). ([arXiv][1])

<a id="ref-openqasm3"></a>
**[16]** A. W. Cross *et al.*, “OpenQASM 3: A broader and deeper quantum assembly language,” arXiv:2104.14722 (2021/2022). ([arXiv][2])

<a id="ref-openqasm3spec"></a>
**[17]** OpenQASM 3.0 Specification (web documentation). ([OpenQASM][3])

<a id="ref-google-below-threshold"></a>
**[18]** Google Quantum AI, “Quantum error correction below the surface code threshold,” *Nature* (2025). ([Nature][4])

<a id="ref-google-scaling"></a>
**[19]** Google Quantum AI, “Suppressing quantum errors by scaling a surface code logical qubit,” *Nature* (2023). ([Nature][5])



<a id="ref-mlir"></a>
**[20]** C. Lattner, M. Amini, U. Bondhugula *et al.*, “MLIR: A Compiler Infrastructure for the End of Moore’s Law,” *arXiv:2002.11054*, 2020.

<a id="ref-fowler-surface"></a>
**[21]** A. G. Fowler, M. Mariantoni, J. M. Martinis, and A. N. Cleland, “Surface codes: Towards practical large-scale quantum computation,” *Physical Review A*, vol. 86, 2012.

<a id="ref-google-scaling"></a>
**[22]** Google Quantum AI, “Suppressing quantum errors by scaling a surface code logical qubit,” *Nature*, 2023.

<a id="ref-google-below-threshold"></a>
**[23]** Google Quantum AI, “Quantum error correction below the surface code threshold,” *Nature*, 2025.

<a id="ref-gidney-ftcosts"></a>
**[24]** A. Gidney and M. Ekerå, “How to factor 2048-bit RSA integers in 8 hours using 20 million noisy qubits,” *Quantum*, vol. 5, 2021.

<a id="ref-eastin-knill"></a>
**[25]** B. Eastin and E. Knill, “Restrictions on Transversal Encoded Quantum Gate Sets,” *Physical Review Letters*, vol. 102, 2009.


<a id="ref-mlir"></a>
**[26]** [MLIR: A Compiler Infrastructure for the End of Moore’s Law](https://arxiv.org/abs/2002.11054) ([arXiv][2])

<a id="ref-openqasm3"></a>
**[27]** [OpenQASM 3: A broader and deeper quantum assembly language](https://arxiv.org/abs/2104.14722) ([arXiv][3])

<a id="ref-openqasm2"></a>
**[28]** [Open Quantum Assembly Language (OpenQASM 2.0 paper)](https://arxiv.org/abs/1707.03429) ([GitHub][4])

<a id="ref-google-below-threshold"></a>
**[29]** [Quantum error correction below the surface code threshold (Nature)](https://www.nature.com/articles/s41586-024-08449-y) ([Nature][1])

<a id="ref-google-scaling"></a>
**[30]** [Suppressing quantum errors by scaling a surface code logical qubit (Nature)](https://www.nature.com/articles/s41586-022-05434-1) ([Nature][5])


<a id="ref-gottesman-qec"></a>
**[31]** D. Gottesman, *Stabilizer Codes and Quantum Error Correction*, PhD Thesis, California Institute of Technology, 1997.

<a id="ref-eastin-knill"></a>
**[32]** B. Eastin and E. Knill, “Restrictions on Transversal Encoded Quantum Gate Sets,” *Physical Review Letters*, vol. 102, 2009.

<a id="ref-google-scaling"></a>
**[33]** Google Quantum AI, “Suppressing quantum errors by scaling a surface code logical qubit,” *Nature*, 2023.

<a id="ref-google-below-threshold"></a>
**[34]** Google Quantum AI, “Quantum error correction below the surface code threshold,” *Nature*, 2025.

<a id="ref-bravyi-kitaev"></a>
**[35]** S. Bravyi and A. Kitaev, “Universal quantum computation with ideal Clifford gates and noisy ancillas,” *Physical Review A*, vol. 71, 2005.

<a id="ref-openqasm3"></a>
**[36]** A. W. Cross *et al.*, “OpenQASM 3: A broader and deeper quantum assembly language,” *ACM Transactions on Quantum Computing*, 2022.


<a id="ref-gottesman-qec"></a>
**[37]** D. Gottesman, *Stabilizer Codes and Quantum Error Correction*, PhD Thesis, California Institute of Technology, 1997.

<a id="ref-eastin-knill"></a>
**[38]** B. Eastin and E. Knill, “Restrictions on Transversal Encoded Quantum Gate Sets,” *Physical Review Letters*, vol. 102, 2009.

<a id="ref-bravyi-kitaev"></a>
**[39]** S. Bravyi and A. Kitaev, “Universal quantum computation with ideal Clifford gates and noisy ancillas,” *Physical Review A*, vol. 71, 2005.

<a id="ref-gidney-ftcosts"></a>
**[40]** A. Gidney and M. Ekerå, “How to factor 2048-bit RSA integers in 8 hours using 20 million noisy qubits,” *Quantum*, vol. 5, 2021.

<a id="ref-fowler-surface"></a>
**[41]** A. G. Fowler *et al.*, “Surface codes: Towards practical large-scale quantum computation,” *Physical Review A*, vol. 86, 2012.

<a id="ref-google-scaling"></a>
**[42]** Google Quantum AI, “Suppressing quantum errors by scaling a surface code logical qubit,” *Nature*, 2023.

<a id="ref-google-below-threshold"></a>
**[43]** Google Quantum AI, “Quantum error correction below the surface code threshold,” *Nature*, 2025.

<a id="ref-openqasm3"></a>
**[44]** A. W. Cross *et al.*, “OpenQASM 3: A broader and deeper quantum assembly language,” *ACM Transactions on Quantum Computing*, 2022.



<a id="ref-mlir"></a>
**[51]** C. Lattner *et al.*, “MLIR: A Compiler Infrastructure for the End of Moore’s Law,” *arXiv:2002.11054*, 2020.

<a id="ref-gottesman-qec"></a>
**[52]** D. Gottesman, *Stabilizer Codes and Quantum Error Correction*, PhD Thesis, California Institute of Technology, 1997.

<a id="ref-google-below-threshold"></a>
**[53]** Google Quantum AI, “Quantum error correction below the surface code threshold,” *Nature*, 2025.

<a id="ref-fowler-surface"></a>
**[54]** A. G. Fowler *et al.*, “Surface codes: Towards practical large-scale quantum computation,” *Physical Review A*, vol. 86, 2012.

<a id="ref-google-scaling"></a>
**[55]** Google Quantum AI, “Suppressing quantum errors by scaling a surface code logical qubit,” *Nature*, 2023.

<a id="ref-ibm-qldpc"></a>
**[56]** S. Bravyi *et al.*, “High-threshold and low-overhead fault-tolerant quantum memory,” *Nature*, 2024.


<a id="ref-mlir"></a>
**[57]** C. Lattner *et al.*, “MLIR: A Compiler Infrastructure for the End of Moore’s Law,” *arXiv:2002.11054*, 2020.

<a id="ref-openqasm3"></a>
**[58]** A. W. Cross *et al.*, “OpenQASM 3: A broader and deeper quantum assembly language,” *ACM Transactions on Quantum Computing*, 2022.

<a id="ref-lao-quantum-ir"></a>
**[59]** L. Lao *et al.*, “Mapping of quantum circuits onto NISQ architectures,” *Quantum Science and Technology*, 2020.
*(Representative of current custom-IR approaches in quantum compilers)*

<a id="ref-quantum-compiler-survey"></a>
**[60]** F. Chong *et al.*, “Programming languages and compiler design for realistic quantum hardware,” *Nature*, 2017.

